# DA6401 Report - Section 2.4 Vanishing Gradient Analysis

Controlled comparison with optimizer fixed to RMSProp:
- ReLU (2x128, 5x128)
- Sigmoid (2x128, 5x128)

Tracks first hidden-layer gradient norms and logs comparison plot/table.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
import wandb

import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

print("Project root:", ROOT)


In [ ]:
from train import train_and_evaluate

RUN_EXPERIMENT = False  # Set to True to launch runs.

PROJECT = "da6401_assignment_1"
ENTITY = None
RUN_GROUP = "report_v1_2_4_vanishing_gradient"

common = dict(
    dataset="mnist",
    epochs=10,
    batch_size=64,
    loss="cross_entropy",
    optimizer="rmsprop",
    learning_rate=0.001,
    weight_decay=0.0,
    weight_init="xavier",
    wandb_project=PROJECT,
    wandb_entity=ENTITY,
    wandb_mode="online",
    seed=42,
)

configs = [
    {"activation": "relu", "num_layers": 2, "hidden_size": [128, 128], "label": "relu_2x128"},
    {"activation": "relu", "num_layers": 5, "hidden_size": [128, 128, 128, 128, 128], "label": "relu_5x128"},
    {"activation": "sigmoid", "num_layers": 2, "hidden_size": [128, 128], "label": "sigmoid_2x128"},
    {"activation": "sigmoid", "num_layers": 5, "hidden_size": [128, 128, 128, 128, 128], "label": "sigmoid_5x128"},
]

out_dir = ROOT / "src" / "tmp"
out_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
results = []

if RUN_EXPERIMENT:
    for cfg in configs:
        args = SimpleNamespace(
            model_path=str(out_dir / f"model_2_4_{cfg['label']}.npy"),
            config_path=str(out_dir / f"config_2_4_{cfg['label']}.json"),
            activation=cfg["activation"],
            num_layers=cfg["num_layers"],
            hidden_size=cfg["hidden_size"],
            **common,
        )

        run = wandb.init(
            project=PROJECT,
            entity=ENTITY,
            name=f"report_2_4_{cfg['label']}",
            group=RUN_GROUP,
            tags=["report_v1", "report_section_2.4", "vanishing_gradient", "mnist", cfg["activation"], f"layers_{cfg['num_layers']}"] ,
            config=vars(args),
        )

        result = train_and_evaluate(args, wandb_run_override=run)
        hist = result["history"]

        grad_curve = [float(ep["grad_norm_first_layer_mean"]) for ep in hist]
        val_loss_curve = [float(ep["val"]["loss"]) for ep in hist]

        rec = {
            "label": cfg["label"],
            "activation": cfg["activation"],
            "num_layers": cfg["num_layers"],
            "run_id": run.id,
            "run_name": run.name,
            "grad_norm_curve": grad_curve,
            "grad_norm_epoch_1": grad_curve[0],
            "grad_norm_epoch_5": grad_curve[4],
            "grad_norm_epoch_10": grad_curve[9],
            "grad_norm_mean": float(np.mean(grad_curve)),
            "val_loss_epoch_10": val_loss_curve[-1],
            "final_val_f1": float(result["final_metrics"]["val"]["f1"]),
            "final_test_f1": float(result["final_metrics"]["test"]["f1"]),
        }
        results.append(rec)

        run.summary["analysis/grad_norm_epoch_1"] = rec["grad_norm_epoch_1"]
        run.summary["analysis/grad_norm_epoch_5"] = rec["grad_norm_epoch_5"]
        run.summary["analysis/grad_norm_epoch_10"] = rec["grad_norm_epoch_10"]
        run.summary["analysis/grad_norm_mean"] = rec["grad_norm_mean"]
        run.finish()

    plt.figure(figsize=(10, 6))
    for rec in results:
        epochs = np.arange(1, len(rec["grad_norm_curve"]) + 1)
        plt.plot(epochs, rec["grad_norm_curve"], marker="o", linewidth=2, label=rec["label"])
    plt.xlabel("Epoch")
    plt.ylabel("First Hidden Layer Gradient Norm")
    plt.title("Section 2.4: Gradient Norm Comparison (RMSProp fixed)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plot_path = out_dir / "report_2_4_gradnorm_plot.png"
    plt.tight_layout()
    plt.savefig(plot_path, dpi=180)
    plt.close()

    summary_run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name="report_2_4_summary",
        group=RUN_GROUP,
        tags=["report_v1", "report_section_2.4", "vanishing_gradient", "summary"],
    )

    table = wandb.Table(columns=[
        "label", "activation", "num_layers", "run_id",
        "grad_norm_epoch_1", "grad_norm_epoch_5", "grad_norm_epoch_10", "grad_norm_mean",
        "val_loss_epoch_10", "final_val_f1", "final_test_f1"
    ])
    for rec in results:
        table.add_data(
            rec["label"], rec["activation"], rec["num_layers"], rec["run_id"],
            rec["grad_norm_epoch_1"], rec["grad_norm_epoch_5"], rec["grad_norm_epoch_10"], rec["grad_norm_mean"],
            rec["val_loss_epoch_10"], rec["final_val_f1"], rec["final_test_f1"],
        )

    summary_run.log({
        "analysis/gradnorm_comparison_table": table,
        "analysis/gradnorm_comparison_plot": wandb.Image(str(plot_path)),
    })
    summary_run.finish()

    payload = {
        "setup": {
            "optimizer": "rmsprop", "dataset": "mnist", "epochs": 10, "batch_size": 64,
            "loss": "cross_entropy", "learning_rate": 0.001, "weight_init": "xavier",
        },
        "runs": results,
        "summary_run_name": "report_2_4_summary",
        "summary_plot_path": str(plot_path),
    }
    with open(out_dir / "report_2_4_vanishing_gradient.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print("Completed 2.4 controlled runs and summary artifacts")
else:
    print("RUN_EXPERIMENT is False. No runs were launched.")


In [ ]:
summary_path = out_dir / "report_2_4_vanishing_gradient.json"
if summary_path.exists():
    data = json.loads(summary_path.read_text())
    for rec in data["runs"]:
        print(rec["label"], rec["run_id"], "g1", round(rec["grad_norm_epoch_1"], 6), "g5", round(rec["grad_norm_epoch_5"], 6), "g10", round(rec["grad_norm_epoch_10"], 6))
else:
    print("No summary file yet. Run experiment cells first.")
